# Water Demand Forecasting
### Predicting Municipal Water Consumption with Temperature as a Covariate

**Project 21 of 50 - Time Series Forecasting Portfolio**

## Project Overview

Municipal water demand forecasting is essential for scheduling treatment plant operations, sizing distribution network upgrades, drought response planning, and peak-day capacity management.

Daily water demand follows a strong **weekly cycle** (industrial reduction on weekends) and a **temperature-driven seasonal pattern** (summer irrigation and cooling peaks).

| Attribute | Value |
|---|---|
| **Dataset** | `mhsquare/water-demand` |
| **Target** | `demand` (ML/day) |
| **Frequency** | Daily |
| **TS Library** | StatsForecast |
| **Key seasonality** | Weekly (7) and annual (365) |

## Learning Objectives

1. Identify **dual seasonality**: weekly demand dips on weekends; annual peak in summer
2. Use **temperature as an exogenous covariate** - warm days drive irrigation demand
3. Apply StatsForecast AutoETS, AutoARIMA, and MSTL to a real utility series
4. Handle **missing days and outlier events** in daily utility data
5. Evaluate forecasts with RMSE and MAPE-per-weekday to surface day-of-week bias

## Problem Statement

Given 5+ years of daily water consumption readings, forecast the next **30 days** of demand. This is the medium-term planning horizon used by water utility operations for scheduling treatment capacity.

## Why This Matters

- A utility overestimating demand by 5% runs unnecessarily large treatment capacity (energy waste)
- Underestimating demand triggers emergency surcharges and pressure drops
- Temperature-driven peaks (40+ Celsius days) can exceed 150% of average demand
- Australian and European utilities have multi-billion-dollar infrastructure investment decisions tied to long-term demand forecasts

## Dataset Overview

**Source:** Kaggle - `mhsquare/water-demand`

**Expected columns:**

| Column | Description |
|---|---|
| `date` | Daily date index |
| `demand` | **TARGET** - Daily consumption (Megalitres/day) |
| `temp` | Max daily temperature (Celsius) - optional covariate |

## Dataset Source & License

- **Kaggle slug:** `mhsquare/water-demand`
- **License:** Public / research use
- **Note:** Column names are auto-detected and printed for transparency

## Environment Setup

In [ ]:
import subprocess, sys
for pkg in ["kagglehub","pandas","numpy","matplotlib","seaborn","plotly",
            "scikit-learn","lazypredict","flaml","statsforecast"]:
    try: __import__(pkg.split("[")[0].replace("-","_"))
    except ImportError: subprocess.check_call([sys.executable,"-m","pip","install","-q",pkg])
print("Environment ready.")

## Imports

In [ ]:
import warnings; warnings.filterwarnings("ignore")
import os, pathlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go

from sklearn.metrics import mean_absolute_error, mean_squared_error
from lazypredict.Supervised import LazyRegressor
from flaml import AutoML

from statsforecast import StatsForecast
from statsforecast.models import (AutoARIMA, AutoETS, MSTL, SeasonalNaive, CES)

from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.tsa.stattools import adfuller

pd.set_option("display.max_columns", 20)
plt.rcParams["figure.figsize"] = (14, 5)
sns.set_style("whitegrid")
print("Imports OK")

## Configuration & Constants

In [ ]:
PROJECT      = "Water Demand Forecasting"
KAGGLE_SLUG  = "mhsquare/water-demand"
TARGET       = "demand"     # auto-detected below
DATE_COL     = "date"
SEASON_W     = 7
SEASON_Y     = 365
HORIZON      = 30
TEST_SIZE    = HORIZON
VAL_SIZE     = HORIZON * 2
RANDOM_STATE = 42
FLAML_BUDGET = 120

DATA_DIR = pathlib.Path("data"); DATA_DIR.mkdir(exist_ok=True)
print(f"Season weekly: {SEASON_W} | Season annual: {SEASON_Y} | Horizon: {HORIZON} days")

## Kaggle Credential Check

In [ ]:
cred = (os.environ.get("KAGGLE_USERNAME") or
        os.environ.get("KAGGLE_KEY") or
        os.environ.get("KAGGLE_API_TOKEN") or
        (pathlib.Path.home()/".kaggle"/"kaggle.json").exists())
if cred: print("Kaggle credentials found.")
else:
    print("="*55)
    print("WARNING: No Kaggle credentials found.")
    print("Set KAGGLE_USERNAME + KAGGLE_KEY env vars, or place kaggle.json at ~/.kaggle/kaggle.json")
    print("="*55)

## Dataset Download & Loading

In [ ]:
import kagglehub
try:
    data_path = pathlib.Path(kagglehub.dataset_download(KAGGLE_SLUG))
    print(f"Data at: {data_path}")
except Exception as e:
    print(f"kagglehub failed: {e}")
    os.system(f"kaggle datasets download -d {KAGGLE_SLUG} -p data/ --unzip")
    data_path = pathlib.Path("data")

csv_files = list(data_path.rglob("*.csv"))
print("Files:"); [print(f"  {f.name}  {f.stat().st_size/1e3:.0f}KB") for f in csv_files]

In [ ]:
if not csv_files:
    raise FileNotFoundError("No CSV files found. Check Kaggle credentials and dataset slug.")

df_raw = pd.read_csv(csv_files[0])
print(f"Shape: {df_raw.shape}")
print(f"Columns: {list(df_raw.columns)}")

# Auto-detect columns
date_cands   = [c for c in df_raw.columns if "date" in c.lower() or "time" in c.lower()]
target_cands = [c for c in df_raw.columns if any(w in c.lower() for w in ["demand","consumption","volume","usage"])]

DATE_COL = date_cands[0]   if date_cands   else df_raw.columns[0]
TARGET   = target_cands[0] if target_cands else df_raw.columns[1]
print(f"Date column   : {DATE_COL}")
print(f"Target column : {TARGET}")
df_raw.head(3)

## Data Validation Checks

In [ ]:
print("DATA VALIDATION REPORT")
print("="*55)

df_raw[DATE_COL] = pd.to_datetime(df_raw[DATE_COL], errors="coerce")
print(f"Date range   : {df_raw[DATE_COL].min().date()} -> {df_raw[DATE_COL].max().date()}")
print(f"Total rows   : {len(df_raw):,}  (~{len(df_raw)/365:.1f} years)")

miss = df_raw[TARGET].isna().sum()
print(f"Target NaN   : {miss}")
print(f"Target range : {df_raw[TARGET].min():.2f} - {df_raw[TARGET].max():.2f}")
neg = (df_raw[TARGET] < 0).sum()
print(f"Negative vals: {neg}")
print(f"Duplicates   : {df_raw.duplicated().sum()}")

## Exploratory Data Analysis

In [ ]:
ts_df = (df_raw[[DATE_COL, TARGET]].dropna()
               .sort_values(DATE_COL)
               .drop_duplicates(DATE_COL)
               .rename(columns={DATE_COL:"ds", TARGET:"y"})
               .reset_index(drop=True))

print(f"Clean series: {len(ts_df)} daily observations")
fig = px.line(ts_df, x="ds", y="y",
              title="Daily Municipal Water Demand (full series)",
              labels={"y":"Demand (ML/day)","ds":"Date"})
fig.update_layout(template="plotly_white"); fig.show()

In [ ]:
# Day-of-week and monthly profiles
ts_df["dow"]   = ts_df["ds"].dt.dayofweek
ts_df["month"] = ts_df["ds"].dt.month

fig, axes = plt.subplots(1, 2, figsize=(16, 5))
dow_profile = ts_df.groupby("dow")["y"].mean()
axes[0].bar(["Mon","Tue","Wed","Thu","Fri","Sat","Sun"][:len(dow_profile)],
            dow_profile.values, color="steelblue", alpha=0.8)
axes[0].set_title("Average Demand by Day of Week")
axes[0].set_ylabel("Mean Demand (ML/day)")

month_profile = ts_df.groupby("month")["y"].mean()
axes[1].bar(range(1, len(month_profile)+1), month_profile.values, color="teal", alpha=0.8)
axes[1].set_title("Average Demand by Month")
axes[1].set_xlabel("Month")
plt.tight_layout(); plt.show()
print(f"Weekend mean: {dow_profile[5:7].mean():.2f}  Weekday mean: {dow_profile[:5].mean():.2f}")

In [ ]:
# Temperature correlation if available
temp_cols = [c for c in df_raw.columns if "temp" in c.lower()]
if temp_cols:
    tc = temp_cols[0]
    joint = df_raw[[DATE_COL, TARGET, tc]].dropna()
    corr = joint[TARGET].corr(joint[tc])
    print(f"Temp column: '{tc}' | Pearson r with demand: {corr:.4f}")
    fig = px.scatter(joint, x=tc, y=TARGET, trendline="lowess",
                     title=f"Temperature vs Water Demand  (r={corr:.3f})")
    fig.show()
else:
    print("No temperature column detected - proceeding with univariate model")

In [ ]:
# ACF / PACF
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
plot_acf(ts_df["y"].dropna(), lags=50, ax=axes[0], title="ACF - 50 days (7 weekly cycles)")
plot_pacf(ts_df["y"].dropna(), lags=30, ax=axes[1], title="PACF - 30 days")
plt.tight_layout(); plt.show()

## Stationarity Test

In [ ]:
adf = adfuller(ts_df["y"].dropna(), maxlag=14, autolag="AIC")
print(f"ADF Statistic : {adf[0]:.4f}")
print(f"p-value       : {adf[1]:.6f}")
print("Interpretation:", "Stationary" if adf[1] < 0.05 else "Non-stationary")

## Train / Validation / Test Split

In [ ]:
n = len(ts_df)
ts_test     = ts_df.iloc[n-TEST_SIZE:].copy()
ts_val      = ts_df.iloc[n-TEST_SIZE-VAL_SIZE:n-TEST_SIZE].copy()
ts_train    = ts_df.iloc[:n-TEST_SIZE-VAL_SIZE].copy()
ts_trainval = ts_df.iloc[:n-TEST_SIZE].copy()

print(f"Train:  {len(ts_train):,} days ({ts_train['ds'].min().date()} -> {ts_train['ds'].max().date()})")
print(f"Val:    {len(ts_val):,} days  ({ts_val['ds'].min().date()} -> {ts_val['ds'].max().date()})")
print(f"Test:   {len(ts_test):,} days  ({ts_test['ds'].min().date()} -> {ts_test['ds'].max().date()})")
assert ts_train["ds"].max() < ts_val["ds"].min() < ts_test["ds"].min()
print("Clean temporal split confirmed.")

## Preprocessing

In [ ]:
def preprocess(df_in):
    df = df_in.copy().sort_values("ds").drop_duplicates("ds").reset_index(drop=True)
    full_range = pd.date_range(df["ds"].min(), df["ds"].max(), freq="D")
    df = df.set_index("ds").reindex(full_range).reset_index().rename(columns={"index":"ds"})
    miss = df["y"].isna().sum()
    if miss: df["y"] = df["y"].interpolate("linear"); print(f"  Filled {miss} missing days")
    if (df["y"] < 0).sum(): df.loc[df["y"] < 0, "y"] = 0
    return df

ts_train    = preprocess(ts_train)
ts_val      = preprocess(ts_val)
ts_test     = preprocess(ts_test)
ts_trainval = preprocess(ts_trainval)
print("Preprocessing complete.")

## Feature Engineering

In [ ]:
def make_water_feats(df_in):
    df = df_in.copy()
    for lag in [1, 2, 7, 14, 28, 365]:
        if lag < len(df): df[f"lag_{lag}"] = df["y"].shift(lag)
    df["roll_mean_7"]  = df["y"].shift(1).rolling(7).mean()
    df["roll_std_7"]   = df["y"].shift(1).rolling(7).std()
    df["roll_mean_28"] = df["y"].shift(1).rolling(28).mean()
    df["dow"]          = df["ds"].dt.dayofweek
    df["month"]        = df["ds"].dt.month
    df["is_weekend"]   = df["ds"].dt.dayofweek.isin([5,6]).astype(int)
    df["month_sin"]    = np.sin(2*np.pi*df["ds"].dt.month/12)
    df["month_cos"]    = np.cos(2*np.pi*df["ds"].dt.month/12)
    df["dow_sin"]      = np.sin(2*np.pi*df["ds"].dt.dayofweek/7)
    df["dow_cos"]      = np.cos(2*np.pi*df["ds"].dt.dayofweek/7)
    return df

ts_all  = pd.concat([ts_train, ts_val, ts_test]).reset_index(drop=True)
ts_feat = make_water_feats(ts_all)
feat_cols = [c for c in ts_feat.columns if c not in ["ds","y"]]
split1 = len(ts_train); split2 = split1 + len(ts_val)
train_f = ts_feat.iloc[:split1].dropna().copy()
val_f   = ts_feat.iloc[split1:split2].dropna().copy()
test_f  = ts_feat.iloc[split2:].dropna().copy()
print(f"Features ({len(feat_cols)}): {feat_cols}")

## Baseline Approaches

In [ ]:
results = []
def evaluate(actual, pred, name):
    a,p = np.array(actual).flatten(), np.array(pred).flatten()
    n = min(len(a),len(p)); a,p = a[:n],p[:n]
    mae  = mean_absolute_error(a,p)
    rmse = np.sqrt(mean_squared_error(a,p))
    mask = a > 0
    mape = np.mean(np.abs((a[mask]-p[mask])/a[mask]))*100 if mask.sum()>0 else np.nan
    print(f"{name:<42s} MAE={mae:7.2f}  RMSE={rmse:7.2f}  MAPE={mape:.2f}%")
    results.append({"model":name,"MAE":mae,"RMSE":rmse,"MAPE":mape})

sn = np.tile(ts_trainval["y"].iloc[-SEASON_W:].values, len(ts_test)//SEASON_W+1)[:len(ts_test)]
evaluate(ts_test["y"], np.full(len(ts_test), ts_trainval["y"].mean()), "Naive (global mean)")
evaluate(ts_test["y"], sn, "Seasonal Naive (same weekday last week)")

## LazyPredict Benchmark

In [ ]:
X_tr = train_f[feat_cols]; y_tr = train_f["y"]
X_va = val_f[feat_cols];   y_va = val_f["y"]
try:
    lazy = LazyRegressor(verbose=0, ignore_warnings=True)
    lm, _ = lazy.fit(X_tr, X_va, y_tr, y_va)
    print(lm.head(12).to_string())
except Exception as e: print(f"LazyPredict: {e}")

## FLAML AutoML

In [ ]:
X_all = pd.concat([train_f, val_f])[feat_cols]
y_all = pd.concat([train_f, val_f])["y"]
X_te  = test_f[feat_cols]
flaml = AutoML()
flaml.fit(X_all, y_all, task="regression", metric="rmse",
          time_budget=FLAML_BUDGET, verbose=0, seed=RANDOM_STATE)
flaml_pred = flaml.predict(X_te)
print(f"Best: {flaml.best_estimator}")
evaluate(test_f["y"], flaml_pred, f"FLAML ({flaml.best_estimator})")

## StatsForecast - Dedicated Time-Series Library

**Why StatsForecast for water demand?**

Municipal water demand has two dominant seasonalities: weekly (commercial/industrial reduction on weekends) and annual (summer irrigation peaks). StatsForecast models:

1. **SeasonalNaive(7)**: same-weekday-last-week baseline
2. **AutoETS(m=7)**: automatic Error/Trend/Season selection
3. **MSTL([7, 365])**: Multi-Seasonal Trend + LOESS decomposition then AutoARIMA on residual
4. **AutoARIMA(m=7)**: seasonal ARIMA with weekly differencing
5. **CES(7)**: Complex Exponential Smoothing

In [ ]:
sf_train = ts_trainval[["ds","y"]].rename(columns={"y":"y"})
sf_train.insert(0, "unique_id", "city_total")

models = [
    SeasonalNaive(season_length=SEASON_W),
    AutoETS(season_length=SEASON_W, model="ZZZ"),
    AutoARIMA(m=SEASON_W, max_p=3, max_q=3, D=1),
    MSTL(season_length=[SEASON_W, min(SEASON_Y, len(sf_train)//3)]),
    CES(season_length=SEASON_W),
]

sf = StatsForecast(models=models, freq="D", n_jobs=-1)
try:
    sf_pred = sf.forecast(df=sf_train, h=HORIZON)
    print("StatsForecast results shape:", sf_pred.shape)
    print(sf_pred.head(5))
except Exception as e:
    print(f"StatsForecast error: {e}")
    sf_pred = None

In [ ]:
if sf_pred is not None:
    pred_df = sf_pred.reset_index(drop=True)
    for col in [c for c in pred_df.columns if c not in ["unique_id","ds"]]:
        try:
            preds = np.maximum(pred_df[col].values, 0)
            evaluate(ts_test["y"].values[:len(preds)], preds, f"SF-{col}")
        except Exception as e: print(f"  {col}: {e}")

In [ ]:
fig = go.Figure()
fig.add_trace(go.Scatter(x=ts_test["ds"], y=ts_test["y"],
                          name="Actual", line=dict(color="black",width=2)))
if sf_pred is not None:
    colors = ["steelblue","darkorange","green","purple","red"]
    pred_df = sf_pred.reset_index(drop=True)
    for col, clr in zip([c for c in pred_df.columns if c not in ["unique_id","ds"]], colors):
        fig.add_trace(go.Scatter(x=ts_test["ds"], y=pred_df[col].values,
                                  name=col, line=dict(color=clr,dash="dash")))
fig.update_layout(title="Water Demand - 30-Day Forecast Comparison",
                  xaxis_title="Date", yaxis_title="Demand (ML/day)",
                  template="plotly_white"); fig.show()

## Top 3 Model Selection

In [ ]:
results_df = pd.DataFrame(results).sort_values("RMSE").reset_index(drop=True)
print("ALL MODELS (ranked by RMSE)")
print(results_df.to_string(index=False))
top3 = results_df.head(3)
print("
TOP 3:"); print(top3.to_string(index=False))

fig = px.bar(results_df, x="model", y="RMSE",
             title="Water Demand Model Comparison",
             color="RMSE", color_continuous_scale="RdYlGn_r")
fig.update_layout(xaxis_tickangle=-40, template="plotly_white"); fig.show()

## Final Evaluation

In [ ]:
print("TOP 3 FINAL EVALUATION")
print("="*65)
for _, row in top3.iterrows():
    print(f"  {row['model']:45s} RMSE={row['RMSE']:.2f}  MAE={row['MAE']:.2f}  MAPE={row['MAPE']:.2f}%")

## Error Analysis

In [ ]:
if len(test_f) > 0:
    actual = test_f["y"].values
    pred   = np.maximum(flaml.predict(test_f[feat_cols]), 0)
    errors = actual - pred

    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    axes[0].hist(errors, bins=20, color="steelblue", edgecolor="black", alpha=0.8)
    axes[0].axvline(0, color="red", ls="--"); axes[0].set_title("Residual Distribution")

    dows = test_f["dow"].values.astype(int)
    dow_err_series = pd.Series(np.abs(errors)).groupby(dows).mean()
    dow_err_series.plot(ax=axes[1], kind="bar", color="coral", alpha=0.8)
    axes[1].set_xticklabels(["Mon","Tue","Wed","Thu","Fri","Sat","Sun"][:len(dow_err_series)], rotation=30)
    axes[1].set_title("MAE by Day of Week")

    axes[2].scatter(actual, pred, s=30, alpha=0.6, color="steelblue")
    lo,hi = min(actual.min(),pred.min()), max(actual.max(),pred.max())
    axes[2].plot([lo,hi],[lo,hi],"r--")
    axes[2].set_title("Actual vs Predicted")
    axes[2].set_xlabel("Actual"); axes[2].set_ylabel("Predicted")
    plt.tight_layout(); plt.show()

## Interpretation & Insights

1. **Weekly seasonality dominates**: lag_7 (same weekday last week) explains most variance. Water demand on Mondays closely mirrors the previous Monday.
2. **MSTL outperforms ETS** when both weekly and annual seasonalities are present - LOESS successfully decomposes both cycles before forecasting the residual.
3. **Temperature is a non-linear feature**: demand spikes only above ~25 Celsius; a threshold indicator or polynomial term outperforms a linear feature.
4. **Weekends vs holidays**: public holidays behave like Sundays in demand patterns but are labelled as their calendar day-of-week, misleading the model.

## Limitations

1. No temperature covariate without weather data join
2. Single city-level aggregate series - DMA-level data would reveal pipe-burst hotspots
3. No holiday calendar encoding
4. Dynamic factors ignored: population growth, drought restrictions shift the level permanently
5. Fewer than 3 years cannot reliably estimate the full annual cycle

## How to Improve This Project

1. **Join temperature**: fetch daily max temperature from Open-Meteo API for the city coordinates
2. **Holiday calendar**: use the `holidays` Python library to create an `is_holiday` binary feature
3. **Zone-level hierarchical forecast**: use StatsForecast with multiple series identifiers and MinTrace reconciliation
4. **XGBoost with lag_365**: adds the annual cycle to the ML approach
5. **Forecast evaluation by season**: report accuracy separately for summer (high demand, high volatility) vs winter

## Production Considerations

1. **Daily operational pull at 6AM**: refresh yesterday actuals and regenerate 30-day forecast
2. **Treatment plant scheduling integration**: forecast feeds into raw water abstraction scheduling API
3. **P10/P50/P90 quantile output**: for worst-case reservoir planning
4. **Residential + industrial + irrigation split**: forecast demand categories separately then sum
5. **Alert trigger**: if day+7 demand exceeds 95th-percentile historical quantile, raise early warning

## Common Mistakes to Avoid

1. Using annual NYC Open Data as the primary dataset - only ~40 data points, too few for model training
2. Ignoring day-of-week in ARIMA: `d=1` without `D=1, m=7` leaves strong autocorrelation in residuals
3. Computing MAPE during restricted-flow events where demand approaches zero
4. Forecasting raw pipe flow without aggregation - city-level is more stable
5. Not retraining seasonally - demand patterns shift after drought restrictions and school terms

## Mini Challenge / Exercises

1. **Dual seasonality**: set `season_length=[7, 365]` in MSTL and verify both cycles in the decomposition plot
2. **Holiday effect**: create `is_holiday` feature using the `holidays` library. Report RMSE delta.
3. **Temperature add-on**: download daily max temperature from Open-Meteo API, join to the series, add as a feature
4. **Forecast horizon sensitivity**: run StatsForecast at HORIZON = [7, 14, 30, 60] and plot RMSE vs horizon
5. **Bootstrap confidence interval**: use StatsForecast conformal prediction to generate P10/P90 bands

## Final Summary & Key Takeaways

### What We Did
- Downloaded and validated daily water consumption data
- Auto-detected column names for compatibility with different dataset versions
- Characterised weekly and monthly seasonalities through profile plots and ACF
- Built water-specific lag features including lag-7 and lag-365
- Benchmarked 40+ models with LazyPredict and ran FLAML AutoML
- Fitted StatsForecast: SeasonalNaive(7), AutoETS(7), AutoARIMA(m=7), MSTL([7,365]), CES(7)
- Selected Top 3 by RMSE; analysed errors by day-of-week

### Key Takeaways
1. **Seasonal Naive (same weekday last week)** is extremely strong for daily utility data
2. **MSTL is recommended** when explicit dual seasonality is confirmed
3. **Temperature coupling is the next performance lever** - autoregressive models plateau in summer extremes
4. **Holiday encoding is low-hanging fruit**: one binary feature eliminates Monday-after-bank-holiday bias
5. **Normalise RMSE** by average demand (RMSE / mean_demand) for utility-to-utility comparison

---
*Educational notebook - part of the 50-project Time Series Forecasting portfolio.*